## Setup

In [4]:
import sys
import os
import importlib

from tabulate import tabulate
import matplotlib.pyplot as plt

sys.path.append("../src")

import utils
import plot
import rstats

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(plot)
importlib.reload(rstats)

<module 'rstats' from '/Users/jsn/dev/nlp-semantic/notebooks/../src/rstats.py'>

In [5]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
DST = f"../results/{BACKEND}/{FILENAME}/"

print(f"{FILENAME=}")
print(f"{BACKEND=}")

df = utils.load(f"../results/{BACKEND}/{FILENAME}/metrics.csv")
print(f"{df.shape=}")

FILENAME='Dados_Italian_2'
BACKEND='gemini'
df.shape=(8520, 14)


In [6]:
tmp = ["id", "category", "concept"] if "category" in df.columns else ["id", "concept"]
grouped = df.groupby(tmp, as_index=False)
dfx = grouped.mean(numeric_only=True)
print(tabulate(dfx.head(3), headers="keys", showindex=False))

id    category    concept       num    distance_next    entropy    distance_centroid_order    distance_centroid_static    MDS1         MDS2    vel_magnitude    acc_magnitude
----  ----------  ---------  ------  ---------------  ---------  -------------------------  --------------------------  ------  -----------  ---------------  ---------------
s100  bird        picchio    3040.5         0.268338        nan                  0.0694995                    0.109895       0  0                   0.73258        nan
s100  bodypart    dito       3079.5         0.278092        nan                  0.0721238                    0.133713       0  0                   0.745776       nan
s100  bodypart    mano       3038           0.121287        nan                  0.0528815                    0.143479       0  2.31296e-18         0.477144         0.727145


## Analysis

In [7]:
metrics = [
    "entropy",
    "distance_next",
    "distance_centroid_static",
    "vel_magnitude",
    "acc_magnitude",
]

if "category" not in dfx.columns:
    dfx = dfx.rename(columns={"concept": "category"})

for i, metric in enumerate(metrics):
    print(f"[{i + 1}/{len(metrics)}] Analyzing '{metric}'")

    # Use lognormal for all metrics, except for distance_centroid_static
    family = "lognormal" if metric != "distance_centroid_static" else "gaussian"
    res, pred, pairs, stdout = plot.boxplot_glmm(dfx, metric, family=family)

    plt.savefig(f"{DST}/boxplot-glmm-{metric}.png", bbox_inches="tight")
    plt.close()

    with open(f"{DST}/r-output-{metric}.txt", "w") as fp:
        fp.write(stdout)

[1/5] Analyzing 'entropy'
[2/5] Analyzing 'distance_next'
[3/5] Analyzing 'distance_centroid_static'
[4/5] Analyzing 'vel_magnitude'
[5/5] Analyzing 'acc_magnitude'
